# 8.11 — Trasformata di Laplace

**Prerequisiti:** 8.10 Trasformata di Fourier (introduzione), 8.4 Equazioni a coefficienti costanti, 5.18 Integrali impropri.


## Perché ci interessa

Nel notebook 8.10 la trasformata di Fourier trasformava la derivazione in moltiplicazione, ma richiedeva funzioni definite su **tutto** $\mathbb R$ e che decadessero abbastanza rapidamente (perché l'integrale improprio converga). Nei problemi di Cauchy (notebook 8.1), tipicamente conosciamo $f(t)$ solo per $t\ge0$ (il "presente e il futuro", non il passato) e le funzioni possono crescere (come $e^{3t}$). La **trasformata di Laplace** risolve entrambi i problemi: integra solo su $[0,+\infty)$ e include un fattore $e^{-st}$ che "smorza" la crescita, rendendo l'integrale convergente per una classe molto più ampia di funzioni — ed è lo strumento **standard** in ingegneria per risolvere equazioni differenziali con condizioni iniziali.

## Teoria, passo dopo passo

### Definizione

Per $f:[0,+\infty)\to\mathbb R$, la **trasformata di Laplace** è:

$$\mathcal L\{f\}(s) = F(s) = \int_0^{+\infty}f(t)e^{-st}\,dt$$

(un integrale improprio, notebook 5.18, convergente per $s$ sufficientemente grande, se $f$ non cresce troppo rapidamente — precisamente, se esistono $M,a$ con $|f(t)|\le Me^{at}$, l'integrale converge per $s>a$).

### Trasformate elementari

- $\mathcal L\{1\}(s)=\displaystyle\int_0^\infty e^{-st}dt=\left[-\dfrac{e^{-st}}s\right]_0^\infty=\dfrac1s$ (per $s>0$; il termine all'infinito si annulla per $s>0$, notebook 5.18).
- $\mathcal L\{e^{at}\}(s)=\displaystyle\int_0^\infty e^{at}e^{-st}dt=\int_0^\infty e^{-(s-a)t}dt=\dfrac1{s-a}$ (per $s>a$, stesso calcolo con $s$ sostituito da $s-a$).
- $\mathcal L\{t\}(s)=\dfrac1{s^2}$, $\mathcal L\{t^n\}(s)=\dfrac{n!}{s^{n+1}}$ (per integrazione per parti ripetuta, notebook 5.17).
- $\mathcal L\{\sin(\omega t)\}(s)=\dfrac\omega{s^2+\omega^2}$, $\mathcal L\{\cos(\omega t)\}(s)=\dfrac s{s^2+\omega^2}$.

### La proprietà fondamentale: trasformare le derivate

> **Teorema.** $\mathcal L\{f'\}(s)=sF(s)-f(0)$.

**Dimostrazione.** $\mathcal L\{f'\}(s)=\displaystyle\int_0^\infty f'(t)e^{-st}dt$. Integriamo per parti (notebook 5.17), con $u=e^{-st},\,dv=f'(t)dt$, quindi $du=-se^{-st}dt,\,v=f(t)$:

$$\int_0^\infty f'(t)e^{-st}dt = \left[f(t)e^{-st}\right]_0^\infty + s\int_0^\infty f(t)e^{-st}dt = (0-f(0)) + sF(s) = sF(s)-f(0)$$

(il termine all'infinito si annulla, assumendo che $f(t)e^{-st}\to0$; il termine in $t=0$ dà $-f(0)e^0=-f(0)$). $\blacksquare$

Applicando due volte: $\mathcal L\{f''\}(s)=s\mathcal L\{f'\}(s)-f'(0)=s(sF(s)-f(0))-f'(0)=s^2F(s)-sf(0)-f'(0)$. Notiamo la differenza cruciale rispetto alla trasformata di Fourier (notebook 8.10): qui compaiono **esplicitamente** i valori iniziali $f(0),f'(0)$ — proprio ciò che serve per un problema di Cauchy!

### Il metodo: trasformare, risolvere algebricamente, antitrasformare

1. Applica $\mathcal L$ a entrambi i membri dell'equazione differenziale, usando le formule sopra: l'equazione differenziale diventa un'equazione **algebrica** per $F(s)$, con $f(0),f'(0)$ già incorporati.
2. Risolvi algebricamente per $F(s)$.
3. **Antitrasforma**: riconosci $F(s)$ come combinazione di trasformate note (spesso dopo aver decomposto in fratti semplici, notebook 1.4/5.17), risalendo a $f(t)$.

In [ ]:
import sympy as sp

t, s = sp.symbols('t s', positive=True)

# Trasformate elementari (verifica diretta)
print("L{1} =", sp.laplace_transform(1, t, s, noconds=True))
print("L{t} =", sp.laplace_transform(t, t, s, noconds=True))
print("L{exp(3t)} =", sp.laplace_transform(sp.exp(3*t), t, s, noconds=True))
print("L{sin(2t)} =", sp.laplace_transform(sp.sin(2*t), t, s, noconds=True))

# Risoluzione di y'+3y=0, y(0)=5, usando Laplace
Y = sp.symbols('Y')
# L{y'} = sY - y(0) = sY - 5
# equazione trasformata: (sY - 5) + 3Y = 0
eq_trasformata = sp.Eq((s*Y - 5) + 3*Y, 0)
Y_soluzione = sp.solve(eq_trasformata, Y)[0]
print("\nY(s) =", Y_soluzione)

y_t = sp.inverse_laplace_transform(Y_soluzione, s, t)
print("y(t) (antitrasformata) =", y_t)

## Esercizio svolto e commentato

**Problema:** risolvi il problema di Cauchy $y''+4y=0$, $y(0)=1$, $y'(0)=0$ usando la trasformata di Laplace.

**Ragionamento:**

**Passo 1 — applichiamo $\mathcal L$ a entrambi i membri**: $\mathcal L\{y''\}+4\mathcal L\{y\}=\mathcal L\{0\}=0$.

**Passo 2 — usiamo la formula per $\mathcal L\{y''\}$** (teoria): $\mathcal L\{y''\}=s^2Y(s)-sy(0)-y'(0)=s^2Y(s)-s\times1-0=s^2Y(s)-s$.

**Passo 3 — sostituiamo nell'equazione trasformata**: $(s^2Y(s)-s)+4Y(s)=0$, cioè $(s^2+4)Y(s)=s$.

**Passo 4 — risolviamo algebricamente per $Y(s)$**: $Y(s)=\dfrac s{s^2+4}$.

**Passo 5 — riconosciamo la forma**: confrontando con la trasformata elementare $\mathcal L\{\cos(\omega t)\}=\dfrac s{s^2+\omega^2}$ (teoria), con $\omega=2$: $Y(s)=\dfrac s{s^2+4}=\mathcal L\{\cos(2t)\}(s)$.

**Conclusione:** $y(t)=\cos(2t)$ (per unicità della trasformata di Laplace — una funzione "abbastanza buona" è univocamente determinata dalla sua trasformata, un fatto analogo all'unicità della serie di Taylor).

**Verifica diretta**: $y(t)=\cos(2t)$, $y''(t)=-4\cos(2t)$; $y''+4y=-4\cos(2t)+4\cos(2t)=0$. ✓ $y(0)=\cos0=1$ ✓, $y'(0)=-2\sin0=0$ ✓.

**Perché il metodo di Laplace, per equazioni a coefficienti costanti come questa, dà "gratuitamente" la soluzione del problema di Cauchy completo, mentre il metodo dell'equazione caratteristica (notebook 8.4) richiedeva un passo separato per imporre le condizioni iniziali:** le condizioni iniziali $y(0),y'(0)$ entrano **direttamente** nella formula di $\mathcal L\{y''\}$ (passo 2), quindi l'equazione algebrica risultante per $Y(s)$ **incorpora già** l'informazione della condizione iniziale — non serve un passo successivo di "trovare $C_1,C_2$" come nel notebook 8.4: la trasformata di Laplace produce direttamente la soluzione **particolare**, non prima la soluzione generale e poi la particolarizzazione.

In [ ]:
import sympy as sp

t, s = sp.symbols('t s', positive=True)
Y = sp.symbols('Y')

eq_trasformata = sp.Eq(s**2 * Y - s*1 - 0 + 4*Y, 0)
Y_sol = sp.solve(eq_trasformata, Y)[0]
print("Y(s) =", Y_sol)

y_t = sp.inverse_laplace_transform(Y_sol, s, t)
print("y(t) =", y_t)

## Esercizi da risolvere

1. Calcola $\mathcal L\{t^2\}(s)$ usando la formula $\mathcal L\{t^n\}=n!/s^{n+1}$.
2. Risolvi $y'-2y=0$, $y(0)=3$ con la trasformata di Laplace.
3. Risolvi $y''-y=0$, $y(0)=0$, $y'(0)=1$ con la trasformata di Laplace (suggerimento: decomponi $Y(s)=\dfrac1{s^2-1}$ in fratti semplici, notebook 1.4).
4. *(più impegnativo)* Risolvi il problema **non omogeneo** $y'+y=e^{2t}$, $y(0)=0$ con la trasformata di Laplace (suggerimento: $\mathcal L\{e^{2t}\}=\frac1{s-2}$; dopo aver trovato $Y(s)$, decomponi in fratti semplici prima di antitrasformare).

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $\mathcal L\{t^2\}=2!/s^3=2/s^3$.

**2.** $\mathcal L\{y'\}-2Y=sY-3-2Y=0\Rightarrow(s-2)Y=3\Rightarrow Y=\dfrac3{s-2}$. Confrontando con $\mathcal L\{e^{at}\}=\frac1{s-a}$: $y(t)=3e^{2t}$.

**3.** $\mathcal L\{y''\}-Y=(s^2Y-0-1)-Y=0\Rightarrow(s^2-1)Y=1\Rightarrow Y=\dfrac1{s^2-1}$. Fratti semplici: $\dfrac1{s^2-1}=\dfrac1{(s-1)(s+1)}=\dfrac{1/2}{s-1}-\dfrac{1/2}{s+1}$. Antitrasformando: $y(t)=\dfrac12e^t-\dfrac12e^{-t}=\sinh t$ (il seno iperbolico, una combinazione di esponenziali analoga a $\sin,\cos$ ma con $\lambda=\pm1$ reali invece che $\pm i$, coerente col Caso 1 del notebook 8.4 anziché il Caso 3).

**4.** $\mathcal L\{y'\}+Y=(sY-0)+Y=(s+1)Y=\dfrac1{s-2}$ (trasformando il termine noto). Quindi $Y=\dfrac1{(s-2)(s+1)}$. Fratti semplici: $\dfrac1{(s-2)(s+1)}=\dfrac{1/3}{s-2}-\dfrac{1/3}{s+1}$. Antitrasformando: $y(t)=\dfrac13e^{2t}-\dfrac13e^{-t}$.

Verifica computazionale:
```python
import sympy as sp

t, s = sp.symbols('t s', positive=True)
Y3 = 1/(s**2-1)
print("Es.3:", sp.inverse_laplace_transform(Y3, s, t))

Y4 = 1/((s-2)*(s+1))
print("Es.4:", sp.inverse_laplace_transform(Y4, s, t))
```

</details>

## Riepilogo finale

- La **trasformata di Laplace** $F(s)=\int_0^\infty f(t)e^{-st}dt$ integra solo su $t\ge0$ e converge per una classe più ampia di funzioni rispetto alla trasformata di Fourier.
- La proprietà chiave $\mathcal L\{f'\}=sF(s)-f(0)$ (e la sua estensione per $f''$) **incorpora direttamente** le condizioni iniziali, trasformando un problema di Cauchy in un'equazione algebrica.
- Il metodo: trasformare l'equazione, risolvere algebricamente per $F(s)$, e antitrasformare (spesso via decomposizione in fratti semplici).
- A differenza del metodo dell'equazione caratteristica (notebook 8.4), la trasformata di Laplace produce **direttamente** la soluzione particolare del problema di Cauchy, senza passare per la soluzione generale.

Questo conclude il **Modulo 8 (Equazioni differenziali)**: dalla definizione di base alle equazioni lineari, dai sistemi matriciali alla stabilità e ai sistemi dinamici, fino agli strumenti di Fourier e Laplace — un percorso che ha riunito e applicato sistematicamente gli strumenti dei Moduli 5 (analisi) e 6 (algebra lineare).

## Collegamenti

Questo è l'ultimo notebook del Modulo 8. Il percorso prosegue con il **Modulo 9 — Numeri complessi e analisi complessa**, che formalizzerà rigorosamente i numeri complessi $e^{i\theta}$ e la formula di Eulero usati informalmente in questo modulo (notebook 8.4, 8.10) per le soluzioni oscillanti.